# Prepare Databricks source data

This notebook prepares the source data and shared helpers used by `01_bronze_ingestion` through `%run ./00_prepare_demo`.

It generates synthetic telemetry and the SQLite fleet roster, loads saved Open-Meteo weather responses, and initializes the dedicated Unity Catalog volume. Bronze, Silver, and Gold transformations run in the downstream notebooks.

Rerunning setup replaces generated source files in the dedicated demo namespace. Weather uses saved responses for reproducibility; no live weather API request is made during this replay.

## Create fictional input data

In [ ]:
"""Make fictional telemetry and a fleet database for the August 2026 demo."""

import json
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
import random
import sqlite3


def generate_sources(output_dir):
    output_dir = Path(output_dir)
    telemetry_dir = output_dir / "telemetry"
    telemetry_dir.mkdir(parents=True, exist_ok=True)
    random_numbers = random.Random(42)
    depots = [
        ("CHI", "Chicago", 41.88, -87.63),
        ("DAL", "Dallas", 32.78, -96.80),
    ]
    vehicles = [
        (f"EV{number:03}", "Model A" if number % 2 else "Model B",
         "CHI" if number <= 12 else "DAL")
        for number in range(1, 25)
    ]
    # This small database represents a static fleet-management snapshot.
    with sqlite3.connect(output_dir / "fleet.sqlite") as database:
        database.execute("PRAGMA foreign_keys = ON")
        database.execute("CREATE TABLE IF NOT EXISTS depots (depot_id TEXT PRIMARY KEY, depot_name TEXT NOT NULL, latitude REAL NOT NULL, longitude REAL NOT NULL)")
        database.execute("CREATE TABLE IF NOT EXISTS vehicles (vehicle_id TEXT PRIMARY KEY, model TEXT NOT NULL, depot_id TEXT NOT NULL REFERENCES depots(depot_id))")
        database.execute("DELETE FROM vehicles")
        database.execute("DELETE FROM depots")
        database.executemany("INSERT INTO depots VALUES (?, ?, ?, ?)", depots)
        database.executemany("INSERT INTO vehicles VALUES (?, ?, ?)", vehicles)

    odometers = {vehicle[0]: 10000.0 + i * 1000 for i, vehicle in enumerate(vehicles)}
    for day_number in range(30):
        day = date(2026, 8, 1) + timedelta(days=day_number)
        readings = []
        for vehicle_id, _, _ in vehicles:
            moving = random_numbers.random() > 0.2
            for reading_number in range(6):
                if reading_number:
                    odometers[vehicle_id] += random_numbers.uniform(4, 18) if moving else 0
                reading = {
                    "vehicle_id": vehicle_id,
                    "event_timestamp": datetime(day.year, day.month, day.day, 8 + 2 * reading_number, tzinfo=timezone.utc).isoformat(),
                    "odometer_km": round(odometers[vehicle_id], 2),
                }
                # A few deliberate problems, each used in a later exercise.
                if vehicle_id == "EV002" and day_number == 2 and reading_number == 2:
                    reading["odometer_km"] = -1.0
                if vehicle_id == "EV003" and day_number == 3 and reading_number == 2:
                    reading["odometer_km"] -= 100.0
                if vehicle_id == "EV006" and day_number == 6:
                    continue  # No telemetry is unknown activity, not zero activity.
                if vehicle_id == "EV007" and day_number == 7 and reading_number > 0:
                    continue
                readings.append(reading)
                if vehicle_id == "EV004" and day_number == 4 and reading_number == 2:
                    readings.append({**reading, "odometer_km": reading["odometer_km"] + 100.0})
                if vehicle_id == "EV005" and day_number == 5 and reading_number == 2:
                    readings.append(reading.copy())
        if day_number == 0:
            readings.extend([
                {"vehicle_id": None, "event_timestamp": "2026-08-01T08:00:00Z", "odometer_km": 100.0},
                {"vehicle_id": "EV008", "event_timestamp": "not-a-time", "odometer_km": 100.0},
                {"vehicle_id": "UNKNOWN", "event_timestamp": "2026-08-01T08:00:00Z", "odometer_km": 10.0},
                {"vehicle_id": "UNKNOWN", "event_timestamp": "2026-08-01T10:00:00Z", "odometer_km": 20.0},
            ])
        path = telemetry_dir / f"{day}.jsonl"
        path.write_text("".join(json.dumps(row) + "\n" for row in readings))
        if day_number == 0:
            with path.open("a") as file:
                file.write('{"vehicle_id": broken json\n')
    print(f"Created 30 telemetry files, 24 vehicles and 2 depots in {output_dir}")




## Weather helper

`parse_daily` converts parallel daily weather arrays into one row per depot and date. Request and cache functions support weather retrieval and snapshot reuse. This Databricks pipeline uses explicit offline replay of saved responses.

In [ ]:
"""Fetch a small, reproducible Open-Meteo snapshot for the two demo depots."""

from datetime import date, datetime, timedelta, timezone
import json
import math
from pathlib import Path
import time

import requests


API_URL = "https://archive-api.open-meteo.com/v1/archive"


def request_weather(parameters):
    # Retry transient HTTP failures, but never turn an outage into missing weather.
    for attempt in range(3):
        try:
            response = requests.get(API_URL, params=parameters, timeout=30)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)
            continue
        if response.status_code == 429 or response.status_code >= 500:
            if attempt < 2:
                time.sleep(2 ** attempt)
                continue
        response.raise_for_status()
        if response.status_code != 200:
            raise ValueError(f"Expected HTTP 200, received {response.status_code}")
        payload = response.json()
        if not isinstance(payload, dict):
            raise ValueError("Weather response must be a JSON object")
        if payload.get("error"):
            raise ValueError(f"Open-Meteo error: {payload.get('reason')}")
        return payload


def parse_daily(payload, start_date, end_date):
    if not isinstance(payload, dict):
        raise ValueError("Weather response must be a JSON object")
    if payload.get("error") or payload.get("utc_offset_seconds") != 0:
        raise ValueError("Weather response must be successful and use UTC days")
    daily = payload.get("daily", {})
    units = payload.get("daily_units", {})
    fields = ["time", "temperature_2m_mean", "precipitation_sum"]
    for field in fields:
        if not isinstance(daily.get(field), list):
            raise ValueError("Weather response is missing daily arrays")

    expected_dates = []
    current_day = start_date
    while current_day <= end_date:
        expected_dates.append(current_day.isoformat())
        current_day = current_day + timedelta(days=1)
    if daily["time"] != expected_dates:
        raise ValueError("Weather dates do not match the requested period")
    for field in fields:
        if len(daily[field]) != len(expected_dates):
            raise ValueError("Weather array lengths do not match the requested period")
    if units.get("temperature_2m_mean") != "°C" or units.get("precipitation_sum") != "mm":
        raise ValueError("Unexpected weather units")
    rows = []
    # The same index in each array describes the same UTC day.
    for i in range(len(expected_dates)):
        day = daily["time"][i]
        temperature = daily["temperature_2m_mean"][i]
        precipitation = daily["precipitation_sum"][i]
        for value in [temperature, precipitation]:
            if value is None:
                continue
            # bool is a Python number subclass, but true/false is not weather.
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                raise ValueError("Weather values must be finite numbers or null")
            if not math.isfinite(value):
                raise ValueError("Weather values must be finite numbers or null")
        if precipitation is not None and precipitation < 0:
            raise ValueError("Precipitation cannot be negative")
        rows.append((date.fromisoformat(day), temperature, precipitation))
    return rows


def load_weather(depots, start_date, end_date, cache_dir, refresh=False, offline=False):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    weather_rows = []
    snapshots = []
    if offline and refresh:
        raise ValueError("Offline mode cannot refresh weather")
    for depot in depots:
        parameters = {
            "latitude": depot["latitude"], "longitude": depot["longitude"],
            "start_date": str(start_date), "end_date": str(end_date),
            "daily": "temperature_2m_mean,precipitation_sum",
            "timezone": "GMT", "temperature_unit": "celsius",
            "precipitation_unit": "mm",
        }
        path = cache_dir / f"{depot['depot_id']}_{start_date}_{end_date}.json"
        if path.exists() and not refresh:
            snapshot = json.loads(path.read_text())
            if snapshot["request"] != parameters:
                raise ValueError("Cached weather request changed; explicitly refresh the snapshot")
            payload = snapshot["response"]
        else:
            if offline:
                raise FileNotFoundError(f"Offline weather snapshot missing: {path}")
            payload = request_weather(parameters)
            snapshot = {"source": API_URL, "request": parameters,
                        "fetched_at": datetime.now(timezone.utc).isoformat(), "response": payload}
        daily_rows = parse_daily(payload, start_date, end_date)
        if not path.exists() or refresh:
            path.write_text(json.dumps(snapshot, indent=2) + "\n")
        for day, temperature, precipitation in daily_rows:
            if temperature is not None:
                temperature = float(temperature)
            if precipitation is not None:
                precipitation = float(precipitation)
            weather_rows.append((depot["depot_id"], day, temperature, precipitation))
        snapshots.append({"file": path.name, "fetched_at": snapshot["fetched_at"], "request": parameters})
    return weather_rows, snapshots


## Create the dedicated volume and copy source files

In [ ]:
from pathlib import Path
import os
import tempfile
import shutil

schema_name = "ev_telematics.fleet_activity_demo_20260910"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {schema_name}.files")
volume_dir = Path("/Volumes/ev_telematics/fleet_activity_demo_20260910/files")
local_source_dir = Path(tempfile.mkdtemp(prefix="fleet_activity_demo_"))
generate_sources(local_source_dir)
shutil.copytree(local_source_dir, volume_dir / "data" / "sources", dirs_exist_ok=True)
os.environ["FLEET_PROJECT_DIR"] = str(volume_dir)
os.environ["FLEET_OFFLINE"] = "1"
os.environ.pop("FLEET_REFRESH_WEATHER", None)
print("Dedicated schema:", schema_name)
print("Source files:", volume_dir / "data" / "sources")

## Load saved real weather responses

The bundled Open-Meteo responses preserve the original request parameters and fetch metadata. They provide reproducible weather inputs without a live API request during the Databricks run.

In [ ]:
# Bundled real responses keep this replay independent of API availability.
snapshot_dir = volume_dir / "examples" / "weather"
snapshot_dir.mkdir(parents=True, exist_ok=True)
(snapshot_dir / 'CHI_2026-08-01_2026-08-30.json').write_text('{\n  "source": "https://archive-api.open-meteo.com/v1/archive",\n  "request": {\n    "latitude": 41.88,\n    "longitude": -87.63,\n    "start_date": "2026-08-01",\n    "end_date": "2026-08-30",\n    "daily": "temperature_2m_mean,precipitation_sum",\n    "timezone": "GMT",\n    "temperature_unit": "celsius",\n    "precipitation_unit": "mm"\n  },\n  "fetched_at": "2026-09-10T14:37:39.964530+00:00",\n  "response": {\n    "latitude": 41.862915,\n    "longitude": -87.64877,\n    "generationtime_ms": 1.750946044921875,\n    "utc_offset_seconds": 0,\n    "timezone": "GMT",\n    "timezone_abbreviation": "GMT",\n    "elevation": 265.0,\n    "daily_units": {\n      "time": "iso8601",\n      "temperature_2m_mean": "\\u00b0C",\n      "precipitation_sum": "mm"\n    },\n    "daily": {\n      "time": [\n        "2026-08-01",\n        "2026-08-02",\n        "2026-08-03",\n        "2026-08-04",\n        "2026-08-05",\n        "2026-08-06",\n        "2026-08-07",\n        "2026-08-08",\n        "2026-08-09",\n        "2026-08-10",\n        "2026-08-11",\n        "2026-08-12",\n        "2026-08-13",\n        "2026-08-14",\n        "2026-08-15",\n        "2026-08-16",\n        "2026-08-17",\n        "2026-08-18",\n        "2026-08-19",\n        "2026-08-20",\n        "2026-08-21",\n        "2026-08-22",\n        "2026-08-23",\n        "2026-08-24",\n        "2026-08-25",\n        "2026-08-26",\n        "2026-08-27",\n        "2026-08-28",\n        "2026-08-29",\n        "2026-08-30"\n      ],\n      "temperature_2m_mean": [\n        21.4,\n        20.8,\n        21.8,\n        24.0,\n        24.1,\n        22.8,\n        23.8,\n        24.5,\n        22.3,\n        22.9,\n        23.1,\n        23.6,\n        23.9,\n        23.4,\n        24.2,\n        24.9,\n        22.8,\n        22.1,\n        21.4,\n        22.0,\n        21.5,\n        22.1,\n        20.6,\n        19.4,\n        20.2,\n        22.8,\n        22.5,\n        22.5,\n        22.3,\n        23.0\n      ],\n      "precipitation_sum": [\n        82.5,\n        3.5,\n        0.0,\n        0.0,\n        8.5,\n        5.0,\n        1.0,\n        0.1,\n        15.9,\n        20.5,\n        19.4,\n        3.9,\n        10.0,\n        8.5,\n        13.2,\n        11.5,\n        3.4,\n        1.7,\n        7.9,\n        0.0,\n        0.0,\n        1.3,\n        0.3,\n        0.0,\n        0.0,\n        2.6,\n        0.7,\n        0.4,\n        0.0,\n        9.0\n      ]\n    }\n  }\n}\n')
(snapshot_dir / 'DAL_2026-08-01_2026-08-30.json').write_text('{\n  "source": "https://archive-api.open-meteo.com/v1/archive",\n  "request": {\n    "latitude": 32.78,\n    "longitude": -96.8,\n    "start_date": "2026-08-01",\n    "end_date": "2026-08-30",\n    "daily": "temperature_2m_mean,precipitation_sum",\n    "timezone": "GMT",\n    "temperature_unit": "celsius",\n    "precipitation_unit": "mm"\n  },\n  "fetched_at": "2026-09-10T14:37:40.842086+00:00",\n  "response": {\n    "latitude": 32.794376,\n    "longitude": -96.8215,\n    "generationtime_ms": 5.050897598266602,\n    "utc_offset_seconds": 0,\n    "timezone": "GMT",\n    "timezone_abbreviation": "GMT",\n    "elevation": 130.0,\n    "daily_units": {\n      "time": "iso8601",\n      "temperature_2m_mean": "\\u00b0C",\n      "precipitation_sum": "mm"\n    },\n    "daily": {\n      "time": [\n        "2026-08-01",\n        "2026-08-02",\n        "2026-08-03",\n        "2026-08-04",\n        "2026-08-05",\n        "2026-08-06",\n        "2026-08-07",\n        "2026-08-08",\n        "2026-08-09",\n        "2026-08-10",\n        "2026-08-11",\n        "2026-08-12",\n        "2026-08-13",\n        "2026-08-14",\n        "2026-08-15",\n        "2026-08-16",\n        "2026-08-17",\n        "2026-08-18",\n        "2026-08-19",\n        "2026-08-20",\n        "2026-08-21",\n        "2026-08-22",\n        "2026-08-23",\n        "2026-08-24",\n        "2026-08-25",\n        "2026-08-26",\n        "2026-08-27",\n        "2026-08-28",\n        "2026-08-29",\n        "2026-08-30"\n      ],\n      "temperature_2m_mean": [\n        34.0,\n        31.0,\n        31.7,\n        33.1,\n        33.3,\n        32.8,\n        33.0,\n        32.0,\n        29.8,\n        30.9,\n        31.4,\n        32.8,\n        33.4,\n        33.0,\n        32.9,\n        33.9,\n        34.5,\n        35.3,\n        34.8,\n        34.7,\n        34.6,\n        34.8,\n        35.3,\n        36.4,\n        34.8,\n        33.8,\n        32.1,\n        32.6,\n        33.2,\n        33.5\n      ],\n      "precipitation_sum": [\n        0.0,\n        0.0,\n        0.0,\n        0.0,\n        0.0,\n        0.0,\n        0.2,\n        4.2,\n        2.9,\n        1.3,\n        0.3,\n        0.0,\n        0.0,\n        0.0,\n        0.5,\n        0.0,\n        0.0,\n        0.0,\n        0.0,\n        0.1,\n        0.3,\n        0.0,\n        0.5,\n        0.0,\n        0.2,\n        1.3,\n        0.9,\n        0.0,\n        0.2,\n        0.0\n      ]\n    }\n  }\n}\n')
print("Weather mode: offline replay of the saved real API responses")